# Load Libraries and Data

In [ ]:
import pandas as pd
import geopandas as gpd
import numpy as np
import seaborn as sns

import matplotlib.pyplot as plt

from shapely.geometry import Point


#Load the cleaned data
clean_rad = pd.read_excel("clean_data_kp.xlsx")

#### Create geodataframe for geopandas

In [ ]:
url = "https://naciscdn.org/naturalearth/110m/cultural/ne_110m_admin_0_countries.zip"
world = gpd.read_file(url)

geometry = [Point(xy) for xy in zip(clean_rad["lon"], clean_rad["lat"])]
geo_rad = gpd.GeoDataFrame(clean_rad, geometry=geometry, crs="EPSG:4326")

### Define a constant colormap to use for all visuals

Default: `managua` 

`managua` is very good for highlighting all points on the graph.  
Change to `Reds` if you would rather only easily see extreme high values. All low values become very hard to see on the light background

In [ ]:
COLOR = "managua"

# How does altitude vary by latitude region?

## Create a column that separates code by latitude region

In [ ]:
geo_rad["region"] = geo_rad["lat"].case_when(
    caselist=[
        (geo_rad["lat"] >= 66.5, "Arctic"),
        (geo_rad["lat"] >= 23.5, "North Temperate"),
        (geo_rad["lat"] >= -23.5, "Equator"),
        (geo_rad["lat"] >= -66.5, "South Temperate"),
        (geo_rad["lat"] >= -200, "Antarctic")
    ]
)

# Define the explicit order you want to see on the plot
desired_order = ['Arctic', 'North Temperate', 'Equator', 'South Temperate', "Antarctic"]

# Convert the column into an ordered categorical category
geo_rad['category'] = pd.Categorical(geo_rad['region'], categories=desired_order, ordered=True)

## Investiagate distribution of altitudes by latitudes

In [ ]:
geo_rad.alt.describe()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6), constrained_layout=True)

geo_rad.boxplot(column="alt", 
                by="category", 
                ax=ax,
                patch_artist=True,
                boxprops=dict(facecolor="steelblue", alpha=0.6),
                medianprops=dict(color="black", linewidth=2),
                flierprops=dict(marker="o", markersize=2, alpha=0.3))

plt.suptitle("")  # removes the automatic "Boxplot grouped by kp" title
# ax.set_yscale("log")
ax.set_title("Altitudes by latitude zone")
ax.set_ylabel("Altitude (km)")
ax.set_xlabel("Latitude Zone")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

world.plot(ax=ax, color="lightgrey")
geo_rad.plot(ax=ax,
            markersize=10,
            alpha=0.5,
            column="alt", 
            cmap=COLOR,
            legend=True,
            legend_kwds = {"label": "Altitude (km)"}
            )

ax.set_aspect("equal", adjustable="box")
plt.title("Altitude of the satellite")
plt.show()

### Analysis of latitude by altitude

I was expecting the altitudes to be highest near the poles, but it appears it's actually second lowest in the Arctic and highest in the Antarctic. Similarly, I was expecting altitude to be lowest at the poles, but it appears it was close to the same as Arctic. I'm not an expert on satellite flight patterns, but this makes me think that the satellite's orbit is eliptical, and it is gaining its speed by dipping in in the Northern Temperate zone. I suspect the satellite has enough speed that it gains some altitude over the Antarctic before crossing over the North Pole

# Split altitude by `in_shadow`
Since the satellite is always moving south when in sunlight and always moving north in shadow, this might allow us to see something interesting about the orbit of the satellite

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3), constrained_layout=True)

for shadow in [0, 1]:
    subset = geo_rad[geo_rad["in_shadow"] == shadow]

    world.plot(ax=axes[shadow], color="lightgrey")
    subset.plot(ax=axes[shadow],
                markersize=10,
                alpha=0.5,
                column="alt", 
                cmap=COLOR,
                legend=True,
                legend_kwds = {"label": "Altitude (km)"}
                )

    # axes[shadow].set_aspect("equal", adjustable="box")
    axes[shadow].set_title(f"Altitude of the satellite when in_shadow == {shadow}")

plt.suptitle("Altitude split by in_shadow", fontsize="xx-large")
plt.show()

## Analysis of splitting altitude by `in_shadow`
I think I might have been right. The altitude while going north dips down and then goes back up when nearing the poles, and then it stays fairly low most of the time it's heading south. I think the satellite might be sort of "falling" as it dips down, gaining speed, and this speed helps it rise back up and continue its orbit.

# View radiation by bins
Bin by 12 km to create 4 bins of equal sizes

## Create a column that bins by altitude

In [ ]:
geo_rad["alt_bin"] = geo_rad["alt"].case_when(
    caselist=[
        (geo_rad["alt"] <= 480, "468 - 480"),
        (geo_rad["alt"] <= 492, "480 - 492"),
        (geo_rad["alt"] <= 504, "492 - 504"),
        (geo_rad["alt"] <= 516, "504 - 516")
    ]
)

# # Define the explicit order you want to see on the plot
# desired_order = ['Arctic', 'North Temperate', 'Equator', 'South Temperate', "Antarctic"]

# # Convert the column into an ordered categorical category
# geo_rad['category'] = pd.Categorical(geo_rad['region'], categories=desired_order, ordered=True)

## Show a boxplot of main radiation binned by altitude

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 4), constrained_layout=True)

for i, rad in enumerate(["xray0_ps", "proton0_ps", "electron0_ps"]):
    geo_rad.boxplot(column=rad, 
                    by="alt_bin", 
                    ax=axes[i],
                    patch_artist=True,
                    boxprops=dict(facecolor="steelblue", alpha=0.6),
                    medianprops=dict(color="black", linewidth=2),
                    flierprops=dict(marker="o", markersize=2, alpha=0.3))
    axes[i].set_yscale("log")
    axes[i].set_title(f"{rad} by altitude")
    axes[i].set_ylabel(f"{rad} (Log)")
    axes[i].set_xlabel("Altitude (km)")

plt.suptitle("Radiation Binned by Altitude", fontsize="xx-large")  # removes the automatic "Boxplot grouped by kp" title
plt.show()

### Analysis of boxplot
- It appears that X-ray radiation becomes lower the higher you go in the atmosphere
    - This brings up something interesting: The times where the satellite is at its highest is when the satellite is over the Antarctic and when it's near the poles in shadow, the times where we've observed the lowest X-ray values. 
    - This brings up an important question: is altitude affecting our X-ray counts, is it overlapping with latitude and sunlight by pure chance, or do all three play a role?
- Protons don't seem to vary quite as much with altitude
- Electrons appear higher the higher in the atmosphere you go

# Investigate effect of altitude vs latitude and in_shadow, focusing on xray. (With AI assistance)

## Correlation analysis

In [ ]:
corr = clean_rad[["lat", "alt", "in_shadow", "xray0_ps", "proton0_ps", "electron0_ps"]].corr(method="spearman")

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr, annot=True, cmap=COLOR, vmin=-1, vmax=1, ax=ax)
ax.xaxis.tick_top()          # move tick marks + labels to the top
plt.show()

In [ ]:
from sklearn.feature_selection import mutual_info_regression
from sklearn.ensemble import RandomForestRegressor

X = geo_rad[['lat', 'alt', 'in_shadow']].dropna()
y = geo_rad.dropna()["xray0_ps"]

mi = mutual_info_regression(X, y)
print(f"Mutual info regression:\n{pd.Series(mi, index=X.columns).sort_values(ascending=False)}\n\n")

rf = RandomForestRegressor(n_estimators=200, random_state=0)
rf.fit(X, y)
print(f"Random Forest Regression:\n{pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)}")

### Analysis of correlation comparison
Both mutual info and random forest regression agree that `alt` is the driving force for `xray0_ps`, while the correlation matrix shows `in_shadow` has the highest correlation. This indicates a more linear relationship with `in_shadow`, and a noticable but nonlinear relationship with `alt`. This makes sense looking at the boxplot of radiations. `xray0_ps` did not decrease linearly as altitude increases.

It's worth noting that all three variables have a moderate strength correlation with `xray0_ps`. 

## Compare altitude at smaller bins

In [ ]:
# group by 1-km bins (or even native altitude if resolution allows)
geo_rad['alt_1km'] = geo_rad['alt'].round(0)  # or use a finer pd.cut if data is dense enough

summary = geo_rad.groupby('alt_1km')['xray0_ps'].agg(['median', 'mean', 'count']).reset_index()

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(summary['alt_1km'], summary['median'], marker='o', markersize=3)
ax.set_yscale('log')
ax.set_xlabel('Altitude (km)')
ax.set_ylabel('xray0_ps median (log)')
ax.set_title('Median xray0_ps vs altitude (fine resolution)')
ax.grid(True, alpha=0.3)

plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(summary['alt_1km'], summary['count'], marker='o', markersize=3)
ax.set_xlabel('Altitude (km)')
ax.set_ylabel('Bin size')
ax.set_title('Median xray0_ps vs bin size (fine resolution)')
ax.grid(True, alpha=0.3)

plt.show()

In [ ]:
# rolling median as a smoother alternative to discrete bins
df_sorted = geo_rad.sort_values('alt')
df_sorted['xray_rolling_median'] = df_sorted['xray0_ps'].rolling(window=200, center=True).median()

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(df_sorted['alt'], df_sorted['xray_rolling_median'])
ax.set_yscale('log')
ax.set_xlabel('Altitude (km)')
ax.set_ylabel('Rolling median xray0_ps (log)')

plt.show()

### Analysis of finer bins
This shows that the dropoff isn't exponential either. It oscillates some. This makes sense given the orbital nature of our data. Because analyzing the data this way involved sorting it out of chronological order, it might be better to look at it in time order

## Compare `xray0_ps` to `alt`, `lat`, and `in_shadow` by `timestamp`

In [ ]:
geo_rad_time_sorted = geo_rad.sort_values('timestamp')

plot_config = [
    ('xray0_ps',  'median', 'xray0_ps',      'Timestamp vs xray_ps',   'log'),
    ('alt',       'median', 'alt (km)',      'Timestamp vs altitude',  'linear'),
    ('lat',       'median', 'lat',           'Timestamp vs latitude',  'linear'),
    ('in_shadow', 'mean',   'in_shadow (frac)', 'Timestamp vs in_shadow', 'linear'),
]

fig, axes = plt.subplots(len(plot_config), 1, figsize=(12, 10), sharex=True)

for ax, (col, agg, ylabel, title, yscale) in zip(axes, plot_config):
    rolled = geo_rad_time_sorted[col].rolling(200, center=True).agg(agg)
    ax.plot(geo_rad_time_sorted['timestamp'], rolled)
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    if yscale == 'log':
        ax.set_yscale('log')
    ax.tick_params(labelbottom=True) # force tick labels on every subplot

axes[-1].set_xlabel('Timestamp')
plt.tight_layout()

### Analysis of `timestamp` vs `xray0_ps`, `alt`, `lat`, and `in_shadow`
A few important things to note:
- The increase in `xray0_ps` with time stays consistent with what we've already observed, meaning the order of the data is intact
- Barring the one spike, altitude appears to be slowly decreasing as time goes on
- At the same time as the altitude spike, latitude dips dramatically and goes much further than we've seen before
- Timestamp and latitude mimic each other's movements surprisngly well until about March

That spike in altitude and dip in latitude tell an interesting story. It almost seems like the satellite deviated from its normal course for some reason. Even more interestingly, X-ray readings don't drop dramatically like you might expect when `alt` goes up and `lat` goes down. This might be worth looking into in greater detail.